# 언어·장르 장벽 분석

**용도** — 출신 언어(글로벌 차트 카테고리 판정)× 타깃 시장 진입률 매트릭스를 계산하고(비영어 히트의 영어권 통과율 23.8%), 표본 산출 과정과 판정 기준을 검증하며, 장르별 통과율 그림을 생성한다.

**보고서 대응** — 보고서 EDA 3 '언어·장르 장벽' 전체(영어권 정의와 표본 · 매트릭스 표 · 그림 4)

**환경** — Python 3.15 / pandas 3.1.1 / scikit-learn 1.9.1 / LightGBM 4.7.0 / SHAP 0.54.1 (requirements.txt)

**실행 순서** — 00(개요) → 01(수집·결합) → 02(EDA) → 03(장르 보완) → 04(언어장벽) → 05(시장파워·미러) → 06(LightGBM) → 07(보정·SHAP) → 08(데뷔)

In [1]:
# 공통 설정 — 데이터는 원본 저장소에서 읽고, 출력물은 이 폴더에만 기록(원본 보호)
from pathlib import Path
import sys
ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
NBOUT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물")
NBOUT_F = NBOUT / "figures"
for _d in [NBOUT, NBOUT_F, NBOUT / "models", NBOUT / "data" / "processed", NBOUT / "data" / "interim"]:
    _d.mkdir(parents=True, exist_ok=True)
print("데이터 루트 =", ROOT)
print("출력 폴더  =", NBOUT)

데이터 루트 = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset
출력 폴더  = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물


**원본 코드** — `scripts/05_language_crossing_recalc.py`

In [2]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 05. 언어 교차 매트릭스(12행) 최종 재계산
#        출신언어 = 글로벌 차트 카테고리(English/Non-English) 판정,
#        혼합 카테고리 8편·국가 차트 미진입 19편 제외 → 영어 1,734 / 비영어 1,824
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/05_language_crossing_recalc.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

g_all = pd.concat([
    pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=","),
    pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")]).drop_duplicates()
c_all = pd.concat([
    pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252"),
    pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")]).drop_duplicates()

cc = pd.read_csv(ROOT / "references/country_continent.csv")
CONT = dict(zip(cc.country_name, cc["대륙"]))
EN_MARKETS = {"United States", "United Kingdom", "Canada", "Australia", "Ireland", "New Zealand"}

hits = set(g_all.show_title.unique())
c_titles = set(c_all.show_title.unique())
g_all["lang_cat"] = g_all.category.str.contains("Non-English").map({True: "비영어", False: "영어"})
lang_sets = g_all.groupby("show_title").lang_cat.apply(set)
only_en = {t for t, s in lang_sets.items() if s == {"영어"}}
only_ne = {t for t, s in lang_sets.items() if s == {"비영어"}}
no_country = hits - c_titles
pop_en = only_en - no_country
pop_ne = only_ne - no_country

ever = c_all.groupby("show_title").country_name.apply(set)

# ---- cmat: 기존 표 구조(6 타깃 그룹 × 2 출신 = 12행) 유지 ----
targets = {"영어권": EN_MARKETS}
for k, name in [("유럽", "유럽(비영어)"), ("아시아", "아시아"), ("남미", "남미"),
                ("아프리카", "아프리카"), ("오세아니아", "오세아니아(비영어)")]:
    targets[name] = {c for c in CONT if CONT[c] == k and c not in EN_MARKETS}

rows = []
for origin, pop in [("영어", pop_en), ("비영어", pop_ne)]:
    pop_ever = ever.loc[ever.index.isin(pop)]
    for tname, grp in targets.items():
        rate = pop_ever.apply(lambda cs: bool(cs & grp)).mean() * 100
        rows.append({"출신언어": origin, "타깃시장": tname, "진입률": round(rate, 1), "표본": len(pop)})
cmat = pd.DataFrame(rows)
cmat.to_csv(NBOUT / "data/processed/language_crossing_matrix.csv", index=False, encoding="utf-8-sig")
print("== cmat (12행) ==")
print(cmat.to_string(index=False))
ne_en = cmat[(cmat.출신언어 == "비영어") & (cmat.타깃시장 == "영어권")].진입률.iloc[0]
en_ne = cmat[(cmat.출신언어 == "영어") & (cmat.타깃시장.isin(["유럽(비영어)", "아시아", "남미", "아프리카", "오세아니아(비영어)"]))].진입률.mean()
print(f"\n지표: 비영어→영어권 {ne_en}% | 영어→비영어권 5그룹 평균 {en_ne:.1f}%")


== cmat (12행) ==
출신언어       타깃시장  진입률   표본
  영어        영어권 93.8 1734
  영어    유럽(비영어) 90.0 1734
  영어        아시아 81.3 1734
  영어         남미 69.2 1734
  영어       아프리카 77.0 1734
  영어 오세아니아(비영어) 54.0 1734
 비영어        영어권 23.8 1824
 비영어    유럽(비영어) 56.4 1824
 비영어        아시아 76.4 1824
 비영어         남미 59.6 1824
 비영어       아프리카 62.2 1824
 비영어 오세아니아(비영어) 32.9 1824

지표: 비영어→영어권 23.8% | 영어→비영어권 5그룹 평균 74.3%


**원본 코드** — `scripts/08_crossing_sample_verify.py`

In [3]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 08. 언어 교차 표본(영어 1,734 / 비영어 1,824, 합 3,558) 산출 과정 재현
#        글로벌 히트 3,585편 중 혼합 카테고리 8편·국가 차트 미진입 19편 제외 검증
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/08_crossing_sample_verify.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

g_old = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=",")
c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
g_new = pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")
c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
g_all = pd.concat([g_old, g_new]).drop_duplicates()
c_all = pd.concat([c_old, c_new]).drop_duplicates()

hits = set(g_all.show_title.unique())
c_titles = set(c_all.show_title.unique())
g_all["lang_cat"] = g_all.category.str.contains("Non-English").map({True: "비영어", False: "영어"})
lang_sets = g_all.groupby("show_title").lang_cat.apply(set)

only_en = {t for t, s in lang_sets.items() if s == {"영어"}}
only_ne = {t for t, s in lang_sets.items() if s == {"비영어"}}
both = {t for t, s in lang_sets.items() if s == {"영어", "비영어"}}
no_country = hits - c_titles
print("국가 차트 미등장:", len(no_country))
print("혼합 카테고리(둘 다 등장):", len(both), sorted(both))

measurable = (only_en | only_ne) - no_country
print("\n영어 카테고리 표본:", len(only_en - no_country))
print("비영어 카테고리 표본:", len(only_ne - no_country))
print("합계:", len(measurable))
print("\n→ cmat 표본: 영어 1,734 / 비영어 1,824 (합 3,558) | 현재 데이터 기준 글로벌 히트", len(hits))


국가 차트 미등장: 19
혼합 카테고리(둘 다 등장): 8 ['Accused', 'Alpha', 'Ballerina', 'Hierarchy', 'Inheritance', "Mother's Day", 'Security', 'The Diplomat']

영어 카테고리 표본: 1734
비영어 카테고리 표본: 1824
합계: 3558

→ cmat 표본: 영어 1,734 / 비영어 1,824 (합 3,558) | 현재 데이터 기준 글로벌 히트 3585


**원본 코드** — `scripts/09_language_category_verify.py`

In [4]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 09. 출신 언어 판정 기준 검증 — 데뷔 국가 기준이 아니라
#        글로벌 차트 카테고리(English/Non-English) 기준임을 입증
#        (데뷔 국가 기준으로는 영어 647/비영어 2,801 — cmat의 1,734/1,824와 불일치)
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/09_language_category_verify.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

g_old = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=",")
c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
g_new = pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")
c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
c_all = pd.concat([c_old, c_new]).drop_duplicates()
g_all = pd.concat([g_old, g_new]).drop_duplicates()

hits = set(g_all.show_title.unique())
c_titles = set(c_all.show_title.unique())
covered = hits & c_titles

EN_MARKETS = {"United States", "United Kingdom", "Canada", "Australia", "Ireland", "New Zealand"}

# 데뷔 국가 기준(반례 검증용)
first_entry = c_all.sort_values("week").groupby("show_title").agg(
    debut_country=("country_name", "first"), debut_week=("week", "first"))
sub = first_entry.loc[first_entry.index.isin(covered)]
lang2 = sub.debut_country.isin(EN_MARKETS).map({True: "영어", False: "비영어"})
print("데뷔 국가 기준:", lang2.value_counts().to_dict(), "← cmat와 불일치 (채택하지 않은 기준)")

# 채택 기준: 글로벌 차트 카테고리
g_all["lang_cat"] = g_all.category.str.contains("Non-English").map({True: "비영어", False: "영어"})
lang_sets = g_all.groupby("show_title").lang_cat.apply(set)
only_en = {t for t, s in lang_sets.items() if s == {"영어"}}
only_ne = {t for t, s in lang_sets.items() if s == {"비영어"}}
both = {t for t, s in lang_sets.items() if s == {"영어", "비영어"}}
no_country = hits - c_titles
pop_en = only_en - no_country
pop_ne = only_ne - no_country
print("\n채택 기준(카테고리): 영어", len(pop_en), "/ 비영어", len(pop_ne), "/ 혼합 제외", len(both),
      "/ 미진입 제외", len(no_country))
print("→ cmat 표본(영어 1,734 / 비영어 1,824)과 일치")


데뷔 국가 기준: {'비영어': 2899, '영어': 667} ← cmat와 불일치 (채택하지 않은 기준)

채택 기준(카테고리): 영어 1734 / 비영어 1824 / 혼합 제외 8 / 미진입 제외 19
→ cmat 표본(영어 1,734 / 비영어 1,824)과 일치


**원본 코드** — `scripts/13_genre_chart.py`

In [5]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 13. 장르별 영어권 통과율 그래프 생성 (18_genre_barrier.png)
#        language_barrier_by_genre.csv 기반 가로 막대그래프 — 전체 평균 23.8% 점선,
#        레드 = 통과율 30% 이상(긴밀감 장르), 회색 = 18% 미만
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, matplotlib 3.11.2 — requirements.txt 참고
#        (한국어 렌더링: Malgun Gothic)
#        실행: python scripts/13_genre_chart.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib
pass  # 노트북 실행용 — Agg 비활성
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
FIG = NBOUT_F

df = pd.read_csv(ROOT / "data/processed/language_barrier_by_genre.csv")
KO = {"Drama": "드라마", "Comedy": "코미디", "Action": "액션", "Thriller": "스릴러", "Crime": "범죄",
      "Romance": "로맨스", "Family": "가족", "Animation": "애니메이션", "Fantasy": "판타지",
      "Documentary": "다큐멘터리", "Reality-TV": "리얼리티", "Sci-Fi": "SF", "Horror": "호러",
      "History": "역사", "Mystery": "미스터리", "Biography": "전기", "Adventure": "어드벤처"}
df["label"] = df.장르.map(lambda g: f"{KO.get(g, g)} ({g})")
df = df.sort_values("영어권진입률", ascending=True)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
fig, ax = plt.subplots(figsize=(8.8, 5.6), dpi=160)
colors = ["#e50914" if r >= 30 else ("#c9c9c9" if r < 18 else "#888888") for r in df.영어권진입률]
ax.barh(range(len(df)), df.영어권진입률, height=0.62, color=colors, zorder=3)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df.label, fontsize=10.5)
ax.set_xlim(0, 40)
ax.set_xlabel("비영어 출신 히트의 영어권 차트 진입률 (%)", fontsize=10, color="#555")
for i, (rate, n_) in enumerate(zip(df.영어권진입률, df.표본)):
    ax.text(rate + 0.3, i, f"{rate:.1f}%  (표본 {n_:,})", va="center", fontsize=9.5,
            fontweight="bold", color="#141414")
ax.axvline(23.8, color="#999", lw=1.4, ls="--", zorder=2)
ax.text(23.9, len(df) - 0.4, "전체 평균 23.8%", fontsize=9.5, color="#666")
ax.grid(axis="x", color="#eee", zorder=0)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#ccc")
ax.tick_params(left=False, bottom=False)
fig.tight_layout()
fig.savefig(FIG / "18_genre_barrier.png", facecolor="white", bbox_inches="tight")
plt.close(fig)
print("그래프 생성 완료: 18_genre_barrier.png")


그래프 생성 완료: 18_genre_barrier.png


### 보고서 인사이트 수치 산출 — 장르별 영어권 통과율 상위·하위

인사이트 4번("미스터리·스릴러·범죄 32~34%는 낮은 벽, 코미디·로맨스 15~16%는 높은 벽")의 근거 수치.

In [6]:
gb = pd.read_csv(ROOT / "data/processed/language_barrier_by_genre.csv")
gb = gb.sort_values("영어권진입률", ascending=False)
print("== 통과율 상위 5 / 하위 3 ==")
print(gb.head(5).to_string(index=False))
print(gb.tail(3).to_string(index=False))
print(f"전체 평균: {gb['영어권진입률'].mean():.1f}%")

== 통과율 상위 5 / 하위 3 ==
       장르  표본  영어권진입률
  Mystery 184    33.7
 Thriller 292    31.8
    Crime 472    31.6
Adventure 153    30.1
   Action 429    28.9
     장르  표본  영어권진입률
Fantasy 112    17.0
Romance 315    15.9
 Comedy 577    14.7
전체 평균: 24.3%
